<a href="https://colab.research.google.com/github/LCaravaggio/Happiness_Polarization/blob/main/Events_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sobre todas las fechas

In [ ]:
# ============================================================
# IDENTIFICACIÓN DE SHOCKS DE POLARIZACIÓN POLÍTICA
# Gemini API
# ============================================================

!pip -q install google-genai pandas

import os
import json
import pandas as pd
from google import genai
from google.colab import userdata

# ------------------------------------------------------------
# 1. API KEY
# ------------------------------------------------------------

API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

# Modelo gratuito
MODEL = "gemini-3.5-flash-lite"


# ------------------------------------------------------------
# 2. PAÍSES
# ------------------------------------------------------------

countries = {
    32: "Argentina",
    68: "Bolivia",
    76: "Brasil",
    152: "Chile",
    170: "Colombia",
    188: "Costa Rica",
    214: "República Dominicana",
    218: "Ecuador",
    222: "El Salvador",
    320: "Guatemala",
    340: "Honduras",
    484: "México",
    558: "Nicaragua",
    591: "Panamá",
    600: "Paraguay",
    604: "Perú",
    858: "Uruguay",
    862: "Venezuela"
}


# ------------------------------------------------------------
# 3. PROMPT
# ------------------------------------------------------------

def build_prompt(country, start_year=1995, end_year=2024):

    return f"""
Tu tarea es construir una base histórica de acontecimientos políticos
que puedan ser considerados SHOCKS DE AUMENTO DE LA POLARIZACIÓN POLÍTICA.

País: {country}
Período: {start_year}-{end_year}

Identifica acontecimientos extraordinarios que hayan aumentado
significativamente la confrontación política, la división entre grupos
políticos o ideológicos, o el conflicto político dentro del país.

NO debes considerar si estos acontecimientos afectaron o podrían haber
afectado el bienestar subjetivo, la felicidad, la satisfacción con la vida
ni ninguna otra variable económica o psicológica.

El criterio debe ser EXCLUSIVAMENTE político.

INCLUYE, cuando corresponda:

- protestas masivas con fuerte confrontación política;
- movimientos sociales nacionales que hayan generado una fuerte
  polarización política;
- huelgas generales con fuerte conflicto político;
- elecciones presidenciales particularmente polarizadas o controversiales;
- elecciones cuyo resultado haya sido fuertemente disputado;
- crisis electorales;
- denuncias importantes de fraude electoral;
- golpes de Estado o intentos de golpe;
- crisis institucionales graves;
- destituciones o intentos de destitución presidencial;
- renuncias presidenciales producidas por crisis políticas;
- referéndums altamente polarizantes;
- conflictos políticos que hayan producido enfrentamientos masivos;
- episodios de violencia política de gran magnitud;
- crisis constitucionales;
- grandes movilizaciones a favor o en contra del gobierno;
- acontecimientos que hayan producido una ruptura política excepcional.

NO INCLUYAS:

- elecciones ordinarias sin características extraordinarias;
- protestas pequeñas o locales;
- manifestaciones rutinarias;
- huelgas sectoriales normales;
- cambios de gobierno normales;
- debates parlamentarios ordinarios;
- escándalos menores;
- acontecimientos cuya importancia sea principalmente económica,
  deportiva, climática o internacional y no tengan una dimensión
  importante de polarización política.

CRITERIOS GENERALES:

1. Sé lo más exhaustivo posible. Es preferible incluir inicialmente
   algún candidato dudoso antes que omitir un acontecimiento político
   importante.

2. Sin embargo, no conviertas la lista en una cronología general de la
   política del país. Los acontecimientos deben ser excepcionales.

3. Identifica la FECHA DE INICIO del shock con la mayor precisión posible.
   Si el acontecimiento tuvo una fecha puntual, utiliza esa fecha.
   Si comenzó durante un período de varios días, utiliza el primer día
   claramente identificable.

4. No inventes fechas. Si no puedes determinar una fecha suficientemente
   precisa, utiliza null.

5. Diferencia entre el acontecimiento y sus consecuencias posteriores.
   Por ejemplo, una protesta que comenzó el 17 de junio debe tener como
   fecha de inicio el 17 de junio, aunque haya continuado durante semanas.

6. Puedes incluir acontecimientos que hayan comenzado antes del período
   indicado solamente si su principal episodio de polarización ocurrió
   dentro del período.

FECHA DEL MOMENTO CULMINANTE:

Además de la fecha de inicio y finalización, identifica `event_day`.

`event_day` debe ser la fecha correspondiente al MOMENTO MÁS IMPORTANTE,
INTENSO O CRÍTICO del acontecimiento desde el punto de vista de la
polarización política.

Debe representar, cuando sea posible:

- el día en que se produjo la mayor movilización;
- el día de mayor confrontación política;
- el día de mayor violencia o enfrentamiento;
- el día en que ocurrió el acontecimiento institucional decisivo;
- el día de una elección o referéndum particularmente polarizado;
- el día en que se produjo el episodio que llevó la crisis a su máximo;
- el día que razonablemente pueda considerarse el "pico" o "súmmum"
  de la crisis.

IMPORTANTE:

`event_day` NO necesariamente coincide con `start_date`.

Por ejemplo, si una ola de protestas comenzó el 10 de junio y alcanzó
su máxima magnitud el 17 de junio:

start_date = 2013-06-10
event_day  = 2013-06-17

Si el acontecimiento tuvo un único día claramente dominante:

start_date = event_day = esa fecha.

Si no es posible determinar razonablemente cuál fue el día culminante,
utiliza null. NO inventes un event_day.

Cuando existan varios días muy importantes, selecciona el que tenga
mayor relevancia política o mayor intensidad dentro del acontecimiento.

El objetivo de `event_day` es disponer de una fecha puntual que pueda
utilizarse posteriormente como fecha del SHOCK en un análisis de eventos.

VARIABLES A DEVOLVER:

Para cada acontecimiento devuelve:

- event_id
- country
- start_date
- end_date
- event_day
- event_type
- intensity
- national_scope
- polarization_mechanism
- description
- confidence

Donde:

intensity:
1 = relativamente importante
2 = importante
3 = muy importante
4 = excepcional
5 = crisis política nacional extraordinaria

national_scope:
"local", "regional", "national"

polarization_mechanism:
explica brevemente por qué el acontecimiento representó un aumento
de la polarización política.

confidence:
"high", "medium", "low"

REGLA ESPECIAL SOBRE FECHAS:

No inventes fechas.

Las tres fechas deben distinguirse conceptualmente:

- start_date = comienzo del acontecimiento;
- end_date = finalización del acontecimiento;
- event_day = momento culminante o de máxima intensidad.

Si solamente puede determinarse con seguridad una fecha, utiliza esa
fecha para start_date y event_day solamente si existe evidencia de que
ese día fue también el momento culminante. En caso contrario, deja
event_day como null.

Devuelve EXCLUSIVAMENTE JSON válido.
No agregues comentarios antes ni después del JSON.
"""


# ------------------------------------------------------------
# 4. FUNCIÓN PARA CONSULTAR GEMINI
# ------------------------------------------------------------

def get_events(country, start_year=1995, end_year=2024):

    prompt = build_prompt(country, start_year, end_year)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 0,
            "response_mime_type": "application/json",
        }
    )

    return json.loads(response.text)

In [ ]:
# ============================================================
# CONSTRUIR BASE DE DATOS DE EVENTOS POLÍTICOS
# ============================================================

import pandas as pd

countries = {
    32: "Argentina",
    68: "Bolivia",
    76: "Brasil",
    152: "Chile",
    170: "Colombia",
    188: "Costa Rica",
    214: "República Dominicana",
    218: "Ecuador",
    222: "El Salvador",
    320: "Guatemala",
    340: "Honduras",
    484: "México",
    558: "Nicaragua",
    591: "Panamá",
    600: "Paraguay",
    604: "Perú",
    858: "Uruguay",
    862: "Venezuela"
}

all_events = []

for country_id, country_name in countries.items():

    print(country_name, flush=True)

    try:
        events = get_events(country_name)

        for event in events:
            event["country_id"] = country_id
            all_events.append(event)

    except Exception as e:
        print(f"ERROR en {country_name}: {e}")

# ------------------------------------------------------------
# DATAFRAME FINAL
# ------------------------------------------------------------

events_df = pd.DataFrame(all_events)

# Fechas
for col in ["start_date", "end_date", "event_day"]:
    if col in events_df.columns:
        events_df[col] = pd.to_datetime(
            events_df[col],
            errors="coerce"
        )

# Orden
events_df = events_df.sort_values(
    ["country_id", "event_day", "start_date"]
).reset_index(drop=True)

print()
print(f"Total de eventos: {len(events_df)}")

Argentina
Bolivia
Brasil
Chile
Colombia
Costa Rica
República Dominicana
Ecuador
El Salvador
Guatemala
Honduras
México
Nicaragua
Panamá
Paraguay
Perú
Uruguay
Venezuela

Total de eventos: 161


In [ ]:
events_df.to_csv('events_LLM.csv')

# Sobre las fechas de las encuestas

In [9]:
# ============================================================
# IDENTIFICACIÓN DE SHOCKS DE POLARIZACIÓN POLÍTICA
# Gemini API
# ============================================================

!pip -q install google-genai pandas

import os
import json
import pandas as pd
from google import genai
from google.colab import userdata

# ------------------------------------------------------------
# 1. API KEY
# ------------------------------------------------------------

API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

# Modelo gratuito
MODEL = "gemini-3.1-flash-lite"


# ------------------------------------------------------------
# 2. PAÍSES
# ------------------------------------------------------------

countries = {
    32: "Argentina",
    68: "Bolivia",
    76: "Brasil",
    152: "Chile",
    170: "Colombia",
    188: "Costa Rica",
    214: "República Dominicana",
    218: "Ecuador",
    222: "El Salvador",
    320: "Guatemala",
    340: "Honduras",
    484: "México",
    558: "Nicaragua",
    591: "Panamá",
    600: "Paraguay",
    604: "Perú",
    858: "Uruguay",
    862: "Venezuela"
}


# ------------------------------------------------------------
# 3. PROMPT
# ------------------------------------------------------------
def build_prompt(country, start_date, end_date):

    return f"""
Tu tarea es construir una base histórica de acontecimientos políticos
que puedan ser considerados SHOCKS DE AUMENTO DE LA POLARIZACIÓN POLÍTICA.

País: {country}

PERÍODO EXACTO DE BÚSQUEDA:
Desde: {start_date}
Hasta: {end_date}

IMPORTANTE:
DEBES BUSCAR Y DEVOLVER ÚNICAMENTE acontecimientos cuyo MOMENTO
CULMINANTE (`event_day`) se encuentre dentro del período indicado.

No incluyas acontecimientos cuyo `event_day` esté fuera de este período,
aunque hayan comenzado antes o hayan continuado después.

Identifica acontecimientos extraordinarios que hayan aumentado
significativamente la confrontación política, la división entre grupos
políticos o ideológicos, o el conflicto político dentro del país.

NO debes considerar si estos acontecimientos afectaron o podrían haber
afectado el bienestar subjetivo, la felicidad, la satisfacción con la vida
ni ninguna otra variable económica o psicológica.

El criterio debe ser EXCLUSIVAMENTE político.

INCLUYE, cuando corresponda:

- protestas masivas con fuerte confrontación política;
- movimientos sociales nacionales que hayan generado una fuerte
  polarización política;
- huelgas generales con fuerte conflicto político;
- elecciones presidenciales particularmente polarizadas o controversiales;
- elecciones cuyo resultado haya sido fuertemente disputado;
- crisis electorales;
- denuncias importantes de fraude electoral;
- golpes de Estado o intentos de golpe;
- crisis institucionales graves;
- destituciones o intentos de destitución presidencial;
- renuncias presidenciales producidas por crisis políticas;
- referéndums altamente polarizantes;
- conflictos políticos que hayan producido enfrentamientos masivos;
- episodios de violencia política de gran magnitud;
- crisis constitucionales;
- grandes movilizaciones a favor o en contra del gobierno;
- acontecimientos que hayan producido una ruptura política excepcional.

NO INCLUYAS:

- elecciones ordinarias sin características extraordinarias;
- protestas pequeñas o locales;
- manifestaciones rutinarias;
- huelgas sectoriales normales;
- cambios de gobierno normales;
- debates parlamentarios ordinarios;
- escándalos menores;
- acontecimientos cuya importancia sea principalmente económica,
  deportiva, climática o internacional y no tengan una dimensión
  importante de polarización política.

CRITERIOS GENERALES:

1. Sé lo más exhaustivo posible. Es preferible incluir inicialmente
   algún candidato dudoso antes que omitir un acontecimiento político
   importante.

2. Sin embargo, no conviertas la lista en una cronología general de la
   política del país. Los acontecimientos deben ser excepcionales.

3. Identifica la FECHA DE INICIO del acontecimiento con la mayor
   precisión posible.

4. No inventes fechas. Si no puedes determinar una fecha suficientemente
   precisa, utiliza null.

5. Diferencia entre el acontecimiento y sus consecuencias posteriores.

6. Un acontecimiento puede haber comenzado antes del período de búsqueda
   o continuar después del período de búsqueda. SIN EMBARGO, SOLO DEBES
   INCLUIRLO SI SU MOMENTO CULMINANTE (`event_day`) ESTÁ DENTRO DEL
   PERÍODO INDICADO.

FECHA DEL MOMENTO CULMINANTE:

Además de la fecha de inicio y finalización, identifica `event_day`.

`event_day` debe ser la fecha correspondiente al MOMENTO MÁS IMPORTANTE,
INTENSO O CRÍTICO del acontecimiento desde el punto de vista de la
polarización política.

Debe representar, cuando sea posible:

- el día de mayor movilización;
- el día de mayor confrontación política;
- el día de mayor violencia o enfrentamiento;
- el día en que ocurrió el acontecimiento institucional decisivo;
- el día de una elección o referéndum particularmente polarizado;
- el día en que se produjo el episodio que llevó la crisis a su máximo;
- el día que razonablemente pueda considerarse el pico de la crisis.

IMPORTANTE:

`event_day` NO necesariamente coincide con `start_date`.

Por ejemplo, si una ola de protestas comenzó el 10 de junio y alcanzó
su máxima magnitud el 17 de junio:

start_date = 2013-06-10
event_day  = 2013-06-17

Si el acontecimiento tuvo un único día claramente dominante:

start_date = event_day = esa fecha.

Si no es posible determinar razonablemente cuál fue el día culminante,
utiliza null. NO inventes un event_day.

RESTRICCIÓN FUNDAMENTAL SOBRE EL PERÍODO:

El valor de `event_day` DEBE estar entre {start_date} y {end_date},
inclusive.

Si un acontecimiento comenzó antes de {start_date} pero su pico ocurrió
dentro del período, puede incluirse.

Si un acontecimiento comenzó dentro del período pero su pico ocurrió
después de {end_date}, NO debe incluirse.

Si el acontecimiento tuvo varios episodios y no puede determinarse cuál
fue el pico dentro del período, utiliza null para `event_day` y evalúa
cuidadosamente si corresponde incluirlo.

El objetivo de `event_day` es disponer de una fecha puntual que pueda
utilizarse posteriormente como fecha del SHOCK en un análisis de eventos.

VARIABLES A DEVOLVER:

Para cada acontecimiento devuelve:

- event_id
- country
- start_date
- end_date
- event_day
- event_type
- intensity
- national_scope
- polarization_mechanism
- description
- confidence

Donde:

intensity:
1 = relativamente importante
2 = importante
3 = muy importante
4 = excepcional
5 = crisis política nacional extraordinaria

national_scope:
"local", "regional", "national"

polarization_mechanism:
explica brevemente por qué el acontecimiento representó un aumento
de la polarización política.

confidence:
"high", "medium", "low"

REGLA ESPECIAL SOBRE FECHAS:

No inventes fechas.

Las tres fechas deben distinguirse conceptualmente:

- start_date = comienzo del acontecimiento;
- end_date = finalización del acontecimiento;
- event_day = momento culminante o de máxima intensidad.

RECUERDA:

El criterio para decidir si un acontecimiento pertenece a esta búsqueda
es que `event_day` esté entre {start_date} y {end_date}.

Devuelve EXCLUSIVAMENTE JSON válido.
No agregues comentarios antes ni después del JSON.
"""

# ------------------------------------------------------------
# 4. FUNCIÓN PARA CONSULTAR GEMINI
# ------------------------------------------------------------

def get_events(country, start_year=1995, end_year=2024):

    prompt = build_prompt(country, start_year, end_year)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 0,
            "response_mime_type": "application/json",
        }
    )

    return json.loads(response.text)

In [10]:
# ============================================================
# CONSTRUIR BASE DE DATOS DE EVENTOS POLÍTICOS
# ============================================================

import pandas as pd

countries = {
    32: "Argentina",
    68: "Bolivia",
    76: "Brasil",
    152: "Chile",
    170: "Colombia",
    188: "Costa Rica",
    214: "República Dominicana",
    218: "Ecuador",
    222: "El Salvador",
    320: "Guatemala",
    340: "Honduras",
    484: "México",
    558: "Nicaragua",
    591: "Panamá",
    600: "Paraguay",
    604: "Perú",
    858: "Uruguay",
    862: "Venezuela"
}

all_events = []

for country_id, country_name in countries.items():

    print(country_name, flush=True)

    try:
        events = get_events(country_name)

        for event in events:
            event["country_id"] = country_id
            all_events.append(event)

    except Exception as e:
        print(f"ERROR en {country_name}: {e}")

# ------------------------------------------------------------
# DATAFRAME FINAL
# ------------------------------------------------------------

events_df = pd.DataFrame(all_events)

# Fechas
for col in ["start_date", "end_date", "event_day"]:
    if col in events_df.columns:
        events_df[col] = pd.to_datetime(
            events_df[col],
            errors="coerce"
        )

# Orden
events_df = events_df.sort_values(
    ["country_id", "event_day", "start_date"]
).reset_index(drop=True)

print()
print(f"Total de eventos: {len(events_df)}")

Argentina
Bolivia
Brasil
ERROR en Brasil: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Chile
Colombia
Costa Rica
República Dominicana
Ecuador
El Salvador
Guatemala
Honduras
México
Nicaragua
Panamá
Paraguay
Perú
Uruguay
Venezuela

Total de eventos: 91


In [11]:
# ------------------------------------------------------------
# REPETIR BRASIL
# ------------------------------------------------------------

country_id = 76
country_name = "Brasil"

print(f"Repitiendo {country_name}...", flush=True)

try:
    brazil_events = get_events(country_name)

    # Agregar country_id
    for event in brazil_events:
        event["country_id"] = country_id

    brazil_df = pd.DataFrame(brazil_events)

    # Fechas
    for col in ["start_date", "end_date", "event_day"]:
        if col in brazil_df.columns:
            brazil_df[col] = pd.to_datetime(
                brazil_df[col],
                errors="coerce"
            )

    # Sumar Brasil al dataframe existente
    events_df = pd.concat(
        [events_df, brazil_df],
        ignore_index=True
    )

    # Ordenar nuevamente
    events_df = events_df.sort_values(
        ["country_id", "event_day", "start_date"]
    ).reset_index(drop=True)

    print(f"Brasil agregado: {len(brazil_df)} eventos")
    print(f"Total de eventos: {len(events_df)}")

except Exception as e:
    print(f"ERROR en Brasil: {e}")

Repitiendo Brasil...
Brasil agregado: 6 eventos
Total de eventos: 97


In [12]:
events_df.to_csv('events_LLM2.csv')